# Assignment 3 — NEO County Economic Maps
**Interactive choropleth maps of Northeast Ohio using Folium + GeoPandas**  
Data: Census ACS 5-Year Estimates 2023 + Census TIGER Shapefiles

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
import folium.features
import requests
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

MAPS_PATH  = r'C:\Users\akash\Downloads\TeamNEO_Research_Portfolio\03_County_Maps\maps'
# Set CENSUS_API_KEY as an environment variable before running:
#   Windows: $env:CENSUS_API_KEY = "your_key"
#   Mac/Linux: export CENSUS_API_KEY="your_key"
CENSUS_KEY = os.environ.get('CENSUS_API_KEY', '')
ACS_URL    = 'https://api.census.gov/data/2023/acs/acs5'
TIGER_URL  = 'https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip'

NEO_FIPS = {
    'Cuyahoga': '035',
    'Summit':   '153',
    'Stark':    '151',
    'Lorain':   '093',
    'Lake':     '085',
    'Medina':   '103',
    'Portage':  '133',
}
REGION_AVG_INC  = 68904
REGION_AVG_2544 = 25.2

# ── STEP 1: PULL CENSUS DATA ────────────────────────────────────────────────
VARIABLES = [
    'NAME',
    'B01003_001E',
    'B19013_001E',
    'B15003_001E',
    'B15003_022E', 'B15003_023E',
    'B15003_024E', 'B15003_025E',
    'B01001_001E',
    'B01001_011E', 'B01001_012E',
    'B01001_013E', 'B01001_014E',
    'B01001_035E', 'B01001_036E',
    'B01001_037E', 'B01001_038E',
    'B23025_002E', 'B23025_005E',
]

resp = requests.get(ACS_URL, params={
    'get': ','.join(VARIABLES),
    'for': f"county:{','.join(NEO_FIPS.values())}",
    'in':  'state:39',
    'key': CENSUS_KEY,
})
resp.raise_for_status()

raw    = resp.json()
raw_df = pd.DataFrame(raw[1:], columns=raw[0])
num    = [c for c in raw[0] if c not in ('NAME', 'state', 'county')]
raw_df[num] = raw_df[num].apply(pd.to_numeric)

raw_df['County']        = raw_df['NAME'].str.replace(' County, Ohio', '', regex=False)
raw_df['bach_plus_pct'] = ((raw_df['B15003_022E'] + raw_df['B15003_023E'] +
                             raw_df['B15003_024E'] + raw_df['B15003_025E'])
                            / raw_df['B15003_001E'] * 100).round(1)
raw_df['pct_25_44']     = ((raw_df['B01001_011E'] + raw_df['B01001_012E'] +
                             raw_df['B01001_013E'] + raw_df['B01001_014E'] +
                             raw_df['B01001_035E'] + raw_df['B01001_036E'] +
                             raw_df['B01001_037E'] + raw_df['B01001_038E'])
                            / raw_df['B01001_001E'] * 100).round(1)
raw_df['unemp_rate']    = (raw_df['B23025_005E'] / raw_df['B23025_002E'] * 100).round(1)
raw_df['GEOID']         = raw_df['state'] + raw_df['county']

cen = raw_df[['County', 'GEOID', 'B19013_001E', 'bach_plus_pct', 'pct_25_44', 'unemp_rate']].rename(
    columns={'B19013_001E': 'median_hh_inc'}
).sort_values('County').reset_index(drop=True)

print('Census data pulled:')
print(cen[['County', 'median_hh_inc', 'bach_plus_pct', 'pct_25_44', 'unemp_rate']].to_string(index=False))


In [2]:
# ── STEP 2: LOAD TIGER SHAPEFILE & MERGE ────────────────────────────────────
print('Loading TIGER county shapefile (first run downloads ~70 MB)...')
counties_us = gpd.read_file(TIGER_URL)
ohio        = counties_us[counties_us['STATEFP'] == '39'].copy()
neo_fips_set = {'39' + v for v in NEO_FIPS.values()}
neo_geo      = ohio[ohio['GEOID'].isin(neo_fips_set)].copy()

# Merge Census data onto geometry
gdf = neo_geo.merge(cen, on='GEOID', how='left')
gdf = gdf.to_crs(epsg=4326)   # Folium needs WGS-84

print(f'GeoDataFrame: {len(gdf)} counties merged')
print(gdf[['County','GEOID','median_hh_inc','unemp_rate','pct_25_44']].to_string(index=False))

Loading TIGER county shapefile (first run downloads ~70 MB)...
GeoDataFrame: 7 counties merged
  County GEOID  median_hh_inc  unemp_rate  pct_25_44
    Lake 39085          77952         4.3       23.9
 Portage 39133          72822         5.3       22.9
   Stark 39151          65740         3.9       24.0
  Lorain 39093          70693         4.4       23.7
Cuyahoga 39035          62823         6.9       26.5
  Summit 39153          71016         5.2       25.7
  Medina 39103          92660         3.0       23.4


In [3]:
# ── HELPERS ──────────────────────────────────────────────────────────────────
def centroid_latlon(gdf_row):
    c = gdf_row.geometry.centroid
    return c.y, c.x

def neo_bounds(gdf):
    b = gdf.total_bounds  # minx miny maxx maxy
    return [[b[1], b[0]], [b[3], b[2]]]

def add_title(m, title, subtitle=None):
    sub = f'<br><span style="font-size:13px;font-weight:normal;color:#555">{subtitle}</span>' if subtitle else ''
    html = f'''
    <div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);
                z-index:1000;background:white;padding:10px 20px;
                border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,0.2);
                text-align:center;max-width:520px">
      <span style="font-size:16px;font-weight:bold;color:#1a3a5c">{title}</span>{sub}
    </div>'''
    m.get_root().html.add_child(folium.Element(html))

def add_source_note(m):
    html = '''<div style="position:fixed;bottom:8px;left:8px;z-index:1000;
                background:rgba(255,255,255,0.85);padding:4px 8px;
                border-radius:4px;font-size:10px;color:#666">
      Source: U.S. Census Bureau, ACS 5-Year Estimates 2023
    </div>'''
    m.get_root().html.add_child(folium.Element(html))

# Map center (centroid of all NEO counties)
center_lat = gdf.geometry.centroid.y.mean()
center_lon = gdf.geometry.centroid.x.mean()
print(f'Map center: {center_lat:.3f}, {center_lon:.3f}')

Map center: 41.318, -81.585


In [4]:
# ── MAP 1: UNEMPLOYMENT RATE ────────────────────────────────────────────────
reg_unemp = (cen['unemp_rate'] * cen['median_hh_inc']).sum() / cen['median_hh_inc'].sum()  # rough avg
reg_unemp = round(cen['unemp_rate'].mean(), 1)

m1 = folium.Map(location=[center_lat, center_lon], zoom_start=9,
                tiles='CartoDB positron', prefer_canvas=True)
add_title(m1, 'Unemployment Rate by County — Northeast Ohio (2023)')
add_source_note(m1)

choropleth1 = folium.Choropleth(
    geo_data=gdf.__geo_interface__,
    data=gdf,
    columns=['GEOID','unemp_rate'],
    key_on='feature.properties.GEOID',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.8,
    line_color='white',
    legend_name='Unemployment Rate (%)',
    nan_fill_color='#f0f0f0',
).add_to(m1)

# Hover tooltip
folium.GeoJson(
    gdf.__geo_interface__,
    style_function=lambda x: {'fillOpacity':0,'color':'transparent','weight':0},
    tooltip=folium.GeoJsonTooltip(
        fields=['County','unemp_rate'],
        aliases=['County','Unemployment Rate (%)'],
        localize=True,
        sticky=True,
        labels=True,
        style='font-size:13px;font-family:Arial;',
    )
).add_to(m1)

# Comparison tooltip overlay (custom)
for _, row in gdf.iterrows():
    lat, lon = centroid_latlon(row)
    diff = row['unemp_rate'] - reg_unemp
    diff_str = f"+{diff:.1f}pp vs region avg" if diff > 0 else f"{diff:.1f}pp vs region avg"
    popup_html = f"""
    <div style='font-family:Arial;min-width:160px'>
      <b style='font-size:14px'>{row['County']}</b><br>
      Unemployment: <b>{row['unemp_rate']}%</b><br>
      Region avg: {reg_unemp:.1f}%<br>
      <span style='color:{"#c0392b" if diff > 0 else "#27ae60"}'>{diff_str}</span>
    </div>"""
    folium.CircleMarker(
        location=[lat, lon],
        radius=3, color='white', fill=True, fill_opacity=0.01, weight=0,
        popup=folium.Popup(popup_html, max_width=220),
    ).add_to(m1)

# Best and worst markers
worst = gdf.loc[gdf['unemp_rate'].idxmax()]
best  = gdf.loc[gdf['unemp_rate'].idxmin()]
for row, icon, color, label in [
    (worst, 'arrow-up',   'red',   f"Highest: {worst['unemp_rate']}%"),
    (best,  'arrow-down', 'green', f"Lowest: {best['unemp_rate']}%"),
]:
    lat, lon = centroid_latlon(row)
    folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(color=color, icon=icon, prefix='fa'),
        popup=folium.Popup(f"<b>{row['County']}</b><br>{label}", max_width=180),
        tooltip=f"{row['County']}: {label}",
    ).add_to(m1)

m1.fit_bounds(neo_bounds(gdf))
path1 = os.path.join(MAPS_PATH, 'neo_map1_unemployment.html')
m1.save(path1)
print(f'Map 1 saved: {path1}')

Map 1 saved: C:\Users\akash\Downloads\TeamNEO_Research_Portfolio\03_County_Maps\maps\neo_map1_unemployment.html


In [5]:
# ── MAP 2: MEDIAN HOUSEHOLD INCOME ─────────────────────────────────────────
m2 = folium.Map(location=[center_lat, center_lon], zoom_start=9,
                tiles='CartoDB positron', prefer_canvas=True)
add_title(m2, 'Median Household Income by County — Northeast Ohio (2023)')
add_source_note(m2)

folium.Choropleth(
    geo_data=gdf.__geo_interface__,
    data=gdf,
    columns=['GEOID','median_hh_inc'],
    key_on='feature.properties.GEOID',
    fill_color='Blues',
    fill_opacity=0.75,
    line_opacity=0.8,
    line_color='white',
    legend_name='Median Household Income ($)',
).add_to(m2)

folium.GeoJson(
    gdf.__geo_interface__,
    style_function=lambda x: {'fillOpacity':0,'color':'transparent','weight':0},
    tooltip=folium.GeoJsonTooltip(
        fields=['County','median_hh_inc'],
        aliases=['County','Median HH Income ($)'],
        localize=True, sticky=True, labels=True,
        style='font-size:13px;font-family:Arial;',
    )
).add_to(m2)

for _, row in gdf.iterrows():
    lat, lon = centroid_latlon(row)
    diff = row['median_hh_inc'] - REGION_AVG_INC
    diff_str = f"+${diff:,.0f} above region avg" if diff >= 0 else f"${abs(diff):,.0f} below region avg"
    popup_html = f"""
    <div style='font-family:Arial;min-width:190px'>
      <b style='font-size:14px'>{row['County']}</b><br>
      Median Income: <b>${row['median_hh_inc']:,}</b><br>
      Region avg: ${REGION_AVG_INC:,}<br>
      <span style='color:{"#27ae60" if diff >= 0 else "#c0392b"}'>{diff_str}</span>
    </div>"""
    folium.CircleMarker(
        location=[lat, lon], radius=3,
        color='white', fill=True, fill_opacity=0.01, weight=0,
        popup=folium.Popup(popup_html, max_width=240),
    ).add_to(m2)

# Highlight Medina (highest) and Cuyahoga (lowest)
medina   = gdf[gdf['County'] == 'Medina'].iloc[0]
cuyahoga = gdf[gdf['County'] == 'Cuyahoga'].iloc[0]
for row, icon, color in [
    (medina,   'star',      'darkblue'),
    (cuyahoga, 'info-sign', 'orange'),
]:
    lat, lon = centroid_latlon(row)
    folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(color=color, icon=icon, prefix='glyphicon'),
        popup=folium.Popup(
            f"<b>{row['County']}</b><br>Income: ${row['median_hh_inc']:,}<br>"
            f"{'Highest' if row['County']=='Medina' else 'Lowest'} in region",
            max_width=200),
        tooltip=f"{row['County']}: ${row['median_hh_inc']:,}",
    ).add_to(m2)

m2.fit_bounds(neo_bounds(gdf))
path2 = os.path.join(MAPS_PATH, 'neo_map2_income.html')
m2.save(path2)
print(f'Map 2 saved: {path2}')

Map 2 saved: C:\Users\akash\Downloads\TeamNEO_Research_Portfolio\03_County_Maps\maps\neo_map2_income.html


In [6]:
# ── MAP 3: BEND THE CURVE — 25-44 WORKFORCE ─────────────────────────────────
m3 = folium.Map(location=[center_lat, center_lon], zoom_start=9,
                tiles='CartoDB positron', prefer_canvas=True)
add_title(m3, 'Prime Workforce (Ages 25–44) — Northeast Ohio (2023)',
          subtitle="Team NEO's Key Retention Metric — 'Bend the Curve'")
add_source_note(m3)

# Red-green diverging around regional average
folium.Choropleth(
    geo_data=gdf.__geo_interface__,
    data=gdf,
    columns=['GEOID','pct_25_44'],
    key_on='feature.properties.GEOID',
    fill_color='RdYlGn',
    fill_opacity=0.78,
    line_opacity=0.8,
    line_color='white',
    legend_name='% Population Ages 25-44',
).add_to(m3)

# Rich 3-metric tooltip
folium.GeoJson(
    gdf.__geo_interface__,
    style_function=lambda x: {'fillOpacity':0,'color':'transparent','weight':0},
    tooltip=folium.GeoJsonTooltip(
        fields=['County','pct_25_44','bach_plus_pct','median_hh_inc'],
        aliases=['County','Ages 25-44 (%)','Bachelor\'s Degree+ (%)','Median Income ($)'],
        localize=True, sticky=True, labels=True,
        style='font-size:13px;font-family:Arial;',
    )
).add_to(m3)

# Popup with full profile for each county
for _, row in gdf.iterrows():
    lat, lon = centroid_latlon(row)
    diff = row['pct_25_44'] - REGION_AVG_2544
    diff_str = f"+{diff:.1f}pp vs region" if diff >= 0 else f"{diff:.1f}pp vs region"
    popup_html = f"""
    <div style='font-family:Arial;min-width:200px'>
      <b style='font-size:15px;color:#1a3a5c'>{row['County']}</b><br><br>
      Ages 25-44: <b style='color:{"#27ae60" if diff >= 0 else "#c0392b"}'>{row['pct_25_44']}%</b>
        <span style='font-size:11px'>({diff_str})</span><br>
      Bachelor\'s+: <b>{row['bach_plus_pct']}%</b><br>
      Median Income: <b>${row['median_hh_inc']:,}</b>
    </div>"""
    folium.CircleMarker(
        location=[lat, lon], radius=3,
        color='white', fill=True, fill_opacity=0.01, weight=0,
        popup=folium.Popup(popup_html, max_width=240),
    ).add_to(m3)

# Dashed circle + popup around Cuyahoga
cuya_row = gdf[gdf['County'] == 'Cuyahoga'].iloc[0]
cuya_lat, cuya_lon = centroid_latlon(cuya_row)
folium.Circle(
    location=[cuya_lat, cuya_lon],
    radius=22000,
    color='#e74c3c',
    weight=2.5,
    fill=False,
    dash_array='8 6',
    popup=folium.Popup(
        "<div style='font-family:Arial;max-width:230px;font-size:12px'>"
        "<b style='color:#c0392b'>Cuyahoga County — Key Opportunity</b><br><br>"
        "Highest young talent concentration (26.5% aged 25-44) "
        "and highest educational attainment (35.9% bach+) in the region — "
        "but <b>lowest median income ($62,823)</b> and highest unemployment.<br><br>"
        "This is the core 'Bend the Curve' target: talent is here; "
        "high-wage job creation is the missing piece."
        "</div>",
        max_width=250
    ),
    tooltip="Click: Cuyahoga — Highest talent, lowest income",
).add_to(m3)

# Cuyahoga star marker
folium.Marker(
    location=[cuya_lat + 0.07, cuya_lon],
    icon=folium.DivIcon(html=(
        '<div style="font-size:11px;font-weight:bold;color:#c0392b;'
        'background:white;padding:3px 6px;border-radius:4px;'
        'border:1.5px solid #c0392b;white-space:nowrap;'
        'box-shadow:0 1px 4px rgba(0,0,0,0.2)">'
        '⭐ Highest Talent<br>Lowest Income</div>'
    ), icon_size=(140, 36), icon_anchor=(70, 0)),
).add_to(m3)

m3.fit_bounds(neo_bounds(gdf))
path3 = os.path.join(MAPS_PATH, 'neo_map3_bend_the_curve.html')
m3.save(path3)
print(f'Map 3 saved: {path3}')

Map 3 saved: C:\Users\akash\Downloads\TeamNEO_Research_Portfolio\03_County_Maps\maps\neo_map3_bend_the_curve.html


In [7]:
# ── STATIC PNG MAPS (matplotlib + geopandas) ────────────────────────────────
gdf_plot = gdf.to_crs(epsg=3734)   # Ohio State Plane South for nice proportions

def save_static_map(gdf, col, title, cmap_name, out_path,
                    fmt_fn=None, note=None, highlight_counties=None):
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    fig.patch.set_facecolor('white')

    vals = gdf[col].values.astype(float)
    vmin, vmax = vals.min(), vals.max()
    cmap = plt.get_cmap(cmap_name)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    gdf.plot(column=col, cmap=cmap_name, linewidth=1.2,
             edgecolor='white', ax=ax, legend=True,
             legend_kwds={'shrink':0.6,'label':title.split(' — ')[0],
                          'orientation':'horizontal','pad':0.02})

    # County name + value labels
    for _, row in gdf.iterrows():
        c = row.geometry.centroid
        val_str = fmt_fn(row[col]) if fmt_fn else str(row[col])
        ax.annotate(f"{row['County']}\n{val_str}",
                    xy=(c.x, c.y), ha='center', va='center',
                    fontsize=8.5, fontweight='bold', color='#111',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none'))

    # Highlight border for specific counties
    if highlight_counties:
        for cname, color in highlight_counties.items():
            sub = gdf[gdf['County'] == cname]
            sub.boundary.plot(ax=ax, color=color, linewidth=2.5)

    ax.set_title(title, fontsize=13, fontweight='bold', pad=12, color='#1a3a5c')
    ax.set_facecolor('#e8eef4')
    ax.axis('off')
    if note:
        fig.text(0.5, 0.02, note, ha='center', fontsize=8, color='#666', style='italic')
    fig.text(0.01, 0.01, 'Source: U.S. Census Bureau, ACS 5-Year Estimates 2023',
             fontsize=7.5, color='#888')
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'PNG saved: {out_path}')

# Map 1 — Unemployment
save_static_map(
    gdf_plot, 'unemp_rate',
    'Unemployment Rate by County — Northeast Ohio (2023)',
    'YlOrRd',
    os.path.join(MAPS_PATH, 'neo_map1_unemployment.png'),
    fmt_fn=lambda v: f'{v:.1f}%',
    highlight_counties={
        gdf_plot.loc[gdf_plot['unemp_rate'].idxmax(), 'County']: '#c0392b',
        gdf_plot.loc[gdf_plot['unemp_rate'].idxmin(), 'County']: '#27ae60',
    }
)

# Map 2 — Income
save_static_map(
    gdf_plot, 'median_hh_inc',
    'Median Household Income by County — Northeast Ohio (2023)',
    'Blues',
    os.path.join(MAPS_PATH, 'neo_map2_income.png'),
    fmt_fn=lambda v: f'${int(v):,}',
    highlight_counties={'Medina': '#1a3a5c', 'Cuyahoga': '#e67e22'}
)

# Map 3 — Bend the Curve
save_static_map(
    gdf_plot, 'pct_25_44',
    "Prime Workforce (Ages 25–44) — 'Bend the Curve' | Northeast Ohio (2023)",
    'RdYlGn',
    os.path.join(MAPS_PATH, 'neo_map3_bend_the_curve.png'),
    fmt_fn=lambda v: f'{v:.1f}%',
    note='Regional avg: 25.2%  |  Green = above avg  |  Red = below avg',
    highlight_counties={'Cuyahoga': '#c0392b'}
)

PNG saved: C:\Users\akash\Downloads\TeamNEO_Research_Portfolio\03_County_Maps\maps\neo_map1_unemployment.png
PNG saved: C:\Users\akash\Downloads\TeamNEO_Research_Portfolio\03_County_Maps\maps\neo_map2_income.png
PNG saved: C:\Users\akash\Downloads\TeamNEO_Research_Portfolio\03_County_Maps\maps\neo_map3_bend_the_curve.png


In [8]:
# ── VERIFY: DIRECTORY TREE ───────────────────────────────────────────────────
import pathlib
root = pathlib.Path(r'C:\Users\akash\Downloads\TeamNEO_Research_Portfolio\03_County_Maps')
print(f'\n{root.name}\\')
for p in sorted(root.rglob('*')):
    depth = len(p.relative_to(root).parts) - 1
    indent = '    ' * depth + ('├── ' if depth > 0 else '')
    size   = f'  ({p.stat().st_size // 1024} KB)' if p.is_file() else ''
    print(f'{"    " * depth}├── {p.name}{size}')
print('\nAll files confirmed.')


03_County_Maps\
├── maps
    ├── neo_map1_unemployment.html  (451 KB)
    ├── neo_map1_unemployment.png  (78 KB)
    ├── neo_map2_income.html  (451 KB)
    ├── neo_map2_income.png  (87 KB)
    ├── neo_map3_bend_the_curve.html  (452 KB)
    ├── neo_map3_bend_the_curve.png  (96 KB)
├── neo_maps.ipynb  (25 KB)

All files confirmed.
